# Hierarchical Model — Live Text Generation Demo

This notebook loads the already-trained hierarchical model from
`../weights/hierarchical_best.pt` and generates names live. No training needed.

**Run all cells top to bottom.** The last cell is the live-generation cell you can
re-run repeatedly during the demo to produce new names on demand.


In [1]:
import torch
import torch.nn.functional as F


## 1. Vocabulary


In [2]:
# Build the vocabulary from the names.txt file.
# Must exactly match how the model was trained.

words = open('../data/names.txt', 'r').read().splitlines()

chars = sorted(list(set(''.join(words))))
s_to_i = {s: i + 1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
i_to_s = {i: s for s, i in s_to_i.items()}
vocab_size = len(i_to_s)

block_size = 8   # must match training

print(f"vocab_size: {vocab_size}")
print(f"block_size: {block_size}")


vocab_size: 27
block_size: 8


## 2. Layer definitions


In [3]:
# Layer class definitions — must match the training notebook exactly.

class Linear:
    def __init__(self, fan_in, fan_out, bias=True):
        self.weight = torch.randn((fan_in, fan_out)) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    def parameters(self):
        return [self.weight] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_mean = torch.zeros(dim)
        self.running_var = torch.ones(dim)
    def __call__(self, x):
        if self.training:
            if x.ndim == 2:
                dim = 0
            elif x.ndim == 3:
                dim = (0, 1)
            xmean = x.mean(dim, keepdim=True)
            xvar = x.var(dim, keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var  = (1 - self.momentum) * self.running_var  + self.momentum * xvar
        return self.out
    def parameters(self):
        return [self.gamma, self.beta]


class Tanh:
    def __call__(self, x):
        self.out = torch.tanh(x)
        return self.out
    def parameters(self):
        return []


class Embedding:
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn((num_embeddings, embedding_dim))
    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out
    def parameters(self):
        return [self.weight]


class FlattenConsecutive:
    def __init__(self, n):
        self.n = n
    def __call__(self, x):
        B, T, C = x.shape
        x = x.view(B, T // self.n, C * self.n)
        if x.shape[1] == 1:
            x = x.squeeze(1)
        self.out = x
        return self.out
    def parameters(self):
        return []


class Sequential:
    def __init__(self, layers):
        self.layers = layers
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        self.out = x
        return self.out
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


## 3. Model architecture


In [4]:
# Construct the model architecture — must match training exactly.

n_embd = 24
n_hidden = 128

model = Sequential([
    Embedding(vocab_size, n_embd),
    FlattenConsecutive(2), Linear(n_embd * 2,  n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden*2,  n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    FlattenConsecutive(2), Linear(n_hidden*2,  n_hidden, bias=False), BatchNorm1d(n_hidden), Tanh(),
    Linear(n_hidden, vocab_size),
])

parameters = model.parameters()
print(f"Architecture has {sum(p.nelement() for p in parameters)} parameters")


Architecture has 76579 parameters


## 4. Load trained weights + BN buffers


In [5]:
# Load trained weights AND BatchNorm buffers from disk.

best_weights_path = "../weights/hierarchical_best().pt"
checkpoint = torch.load(best_weights_path, map_location='cpu')

# 1. Restore trainable parameters
saved_params = checkpoint["parameters"]
assert len(saved_params) == len(parameters), (
    f"Mismatch: checkpoint has {len(saved_params)} tensors, "
    f"model expects {len(parameters)}."
)
for i, (p, saved) in enumerate(zip(parameters, saved_params)):
    assert p.shape == saved.shape, f"param {i}: shape {p.shape} vs {saved.shape}"
for p, saved in zip(parameters, saved_params):
    p.data = saved.data

# 2. Restore BatchNorm running buffers — this is what was missing before.
saved_bn = checkpoint["bn_buffers"]
bn_layers = [layer for layer in model.layers if isinstance(layer, BatchNorm1d)]
assert len(saved_bn) == len(bn_layers), (
    f"Mismatch: checkpoint has {len(saved_bn)} BN buffers, model has {len(bn_layers)}."
)
for layer, buf in zip(bn_layers, saved_bn):
    layer.running_mean = buf["running_mean"]
    layer.running_var  = buf["running_var"]

# 3. Switch to eval mode
for layer in model.layers:
    layer.training = False

meta = checkpoint.get("metadata", {})
print(f"Loaded checkpoint from {best_weights_path}")
if meta:
    step = meta.get('step', '?')
    vloss = meta.get('val_loss', None)
    vloss_str = f"{vloss:.4f}" if isinstance(vloss, float) else str(vloss)
    print(f"  trained for {step} steps, val_loss={vloss_str}")
print("Model is in eval mode and ready to generate.")


Loaded checkpoint from ../weights/hierarchical_best().pt
  trained for 180000 steps, val_loss=1.9903
Model is in eval mode and ready to generate.


## 5. Generation helper


In [6]:
# Generation helper.

@torch.no_grad()
def generate_name(seed=None):
    """Generate a single name. If seed is given, output is reproducible."""
    g = torch.Generator().manual_seed(seed) if seed is not None else None

    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        if g is not None:
            ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        else:
            ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    return ''.join(i_to_s[i] for i in out if i != 0)


## 6. Live demo — re-run this cell repeatedly


In [12]:
# ===== LIVE DEMO CELL =====
# Re-run this cell repeatedly. Each run produces fresh names.

n_names = 10
print(f"Generating {n_names} names:\n")
for i in range(n_names):
    print(f"  {i+1:2d}. {generate_name()}")


Generating 10 names:

   1. akylah
   2. amah
   3. neela
   4. avraeh
   5. xaydon
   6. abad
   7. saru
   8. leasia
   9. loriana
  10. davynn


## 7. Reproducible generation (for rehearsal)


In [8]:
# Optional: reproducible generation for rehearsal.

print("Seeded (reproducible) sample:")
for i in range(5):
    print(f"  {i+1}. {generate_name(seed=2147483647 + i)}")


Seeded (reproducible) sample:
  1. celia
  2. irish
  3. taylor
  4. victorious
  5. avro


## Notes

- Model trained in `file5.ipynb`, weights at `../weights/hierarchical_best.pt`.
- Checkpoint stores trainable params AND BatchNorm running buffers, so inference
  matches training statistics.
- This notebook does not retrain.
- Layer classes and architecture must match training exactly.
